# Research Proposal: Modeling Daily Parking Ticket Counts in Milwaukee

### John Eckberg

## Overview
[Milwaukee plans 65,000 more parking tickets, longer meter hours to fill transportation budget gap](https://www.jsonline.com/story/news/local/milwaukee/2025/10/21/milwaukee-aims-for-550k-parking-tickets-longer-meter-hours-in-2026/86817755007/) The city of Milwaukee relies on parking ticket revenue in order to fill budget gaps, a pressing issue given the [current deficit](https://urbanmilwaukee.com/2025/03/12/mke-county-county-faces-47-million-2026-budget-deficit/) the county is facing.
This project plans to examine how daily parking ticket activity in Milwaukee varies across time and environmental conditions. Using parking citation records obtained through a Freedom of Information Act request (2014–2022), I propose to model daily ticket counts and evaluate whether temporal, weather, and holiday-related factors explain variation in enforcement volume.

## Research Question
What is the daily rate of parking tickets issued in Milwaukee, and which factors most strongly influence that rate (if any)? In particular, I propose to test whether day of week, month/seasonality, holidays, and precipitation are associated with higher or lower daily ticket counts, including by citation type.

## Hypothesis
- Null Hypothesis: day of week, month/seasonality, holidays, and precipitation do not affect daily ticket counts.
- Alternative Hypothesis: at least one temporal or environmental factor affects daily ticket counts.

## Data Sources
- **Parking citations:** Milwaukee parking ticket records, initially loaded for 2014, with fields including `ISSUENO`, `ISSUEDATE`, `LOCATIONDESC1`, and `VIODESCRIPTION`.
- **Holiday data:** U.S. holiday dates (2004–2021) pulled from Kaggle to identify atypical traffic/parking demand days.
- **Weather data:** Open-Meteo daily weather for Milwaukee, including maximum temperature, minimum temperature, precipitation, and snowfall.


## Method
The analysis would likely focus on a geographically meaningful subset of Milwaukee (e.g., downtown, Lower East Side, Deer District). Because the response variable is a daily count, I propose that the initial model choice would be a Poisson regression GLM.

### Currently Planned Features
- **Calendar effects:** day of week, month, weekend indicator, and potential seasonal terms.
- **Cyclical time encoding:** sine/cosine transforms for month and day-of-week to preserve circular structure.
- **Holiday effects:** binary `is_holiday` indicator, one-hot encoding for major holiday names.
- **Weather exposure:** `max_temp`, `min_temp`, `precipitation`, and `snowfall`; optional threshold features (e.g., heavy precipitation day).
- **Autocorrelation:** lagged ticket count features (e.g., 1-day and 7-day lags) and short rolling averages.



In [1]:
import pandas as pd
import requests
import kagglehub 
import openpyxl

c:\Users\mathewsj\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading in the parking data, just one year

# tickets_df = pd.read_excel("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/tickets_csv/2014_parking_citations_issued_open_records_request.xlsx")
tickets_df = pd.read_excel("data/tickets_csv/2014_parking_citations_issued_open_records_request.xlsx")

tickets_df.head(10)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION
0,474576465,2014-01-01,2444 PALMER ST,UNREGISTERED/ IMPROPERLY REGISTERED VEHICLE
1,474311040,2014-01-01,5419 GALENA ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
2,474311062,2014-01-01,2371 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
3,474353456,2014-01-01,1112 PLEASANT ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
4,474365522,2014-01-01,2969 46TH ST,PARKING PROHIBITED BY OFFICIAL SIGN
5,474365533,2014-01-01,3422 49TH ST,PARKED IN CROSSWALK
6,474311106,2014-01-01,2648 56TH ST,IMPROPERLY DISPLAYED VEHICLE REGISTRATION
7,474417915,2014-01-01,2151 30TH ST,TOW-AWAY ZONE (BLOCKING DRIVEWAY)
8,474455380,2014-01-01,523 34TH ST,PARKING IN EXCESS OF 24 HOURS
9,474468400,2014-01-01,833 HAWLEY RD,PARKED POSTED PRIVATE PROPERTY


In [3]:
# load in holiday data (https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021)

path = kagglehub.dataset_download("donnetew/us-holiday-dates-2004-2021")

print("Path to dataset files:", path)

100%|██████████| 2.45k/2.45k [00:00<00:00, 2.51MB/s]

Extracting files...
Path to dataset files: C:\Users\mathewsj\.cache\kagglehub\datasets\donnetew\us-holiday-dates-2004-2021\versions\1


In [5]:
#holidays_df = pd.read_csv("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/US Holiday Dates (2004-2021).csv")
holidays_df = pd.read_csv("data/US Holiday Dates (2004-2021).csv")

holidays_df.head(10)

,Date,Holiday,WeekDay,Month,Day,Year
0,2004-07-04,4th of July,Sunday,7,4,2004
1,2005-07-04,4th of July,Monday,7,4,2005
2,2006-07-04,4th of July,Tuesday,7,4,2006
3,2007-07-04,4th of July,Wednesday,7,4,2007
4,2008-07-04,4th of July,Friday,7,4,2008
5,2009-07-04,4th of July,Saturday,7,4,2009
6,2010-07-04,4th of July,Sunday,7,4,2010
7,2011-07-04,4th of July,Monday,7,4,2011
8,2012-07-04,4th of July,Wednesday,7,4,2012
9,2013-07-04,4th of July,Thursday,7,4,2013


In [6]:
# weather data (https://open-meteo.com/)
test_data = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=43.0389&longitude=-87.9065&start_date=2014-01-01&end_date=2014-12-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum&timezone=America%2FChicago")
test_data_json = test_data.json()

# minor cleaning of the weather data to get it into a dataframe
weather_df = pd.DataFrame(test_data_json['daily'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.rename(columns={'time': 'date', 'temperature_2m_max': 'max_temp', 'temperature_2m_min': 'min_temp', 'precipitation_sum': 'precipitation','snowfall_sum': 'snowfall'})
weather_df.head(10)


,date,max_temp,min_temp,precipitation,snowfall
0,2014-01-01,-7.6,-13.3,1.3,1.33
1,2014-01-02,-8.2,-13.7,0.7,1.05
2,2014-01-03,-6.6,-17.2,0.0,0.00
3,2014-01-04,0.3,-6.1,0.7,0.91
4,2014-01-05,-4.3,-16.6,1.5,1.40
5,2014-01-06,-17.6,-25.0,0.0,0.21
6,2014-01-07,-15.9,-24.4,0.0,0.00
7,2014-01-08,-12.0,-17.4,0.0,0.00
8,2014-01-09,-3.9,-16.5,0.0,0.14
9,2014-01-10,3.1,-3.8,13.4,0.14


In [8]:
# clean and align date keys before joining
tickets_clean = tickets_df.copy()
holidays_clean = holidays_df.copy()
weather_clean = weather_df.copy()

tickets_clean['join_date'] = pd.to_datetime(tickets_clean['ISSUEDATE'], errors='coerce').dt.normalize()
holidays_clean['join_date'] = pd.to_datetime(holidays_clean['Date'], errors='coerce').dt.normalize()
weather_clean['join_date'] = pd.to_datetime(weather_clean['date'], errors='coerce').dt.normalize()

# merge tickets with holidays and weather on a consistent datetime key
combined_tickets_and_holidays_df = pd.merge(
    tickets_clean,
    holidays_clean,
    how='inner', # in practice should be a left join
    on='join_date'
 )
combined_all_df = pd.merge(
    combined_tickets_and_holidays_df,
    weather_clean,
    how='left',
    on='join_date',
    suffixes=('_holiday', '_weather')
 )

# combined_all_df = combined_all_df.drop(columns=['join_date','Date','date','ISSUEDATE'])
# combined_all_df.head(10)

# sample of the combined dataframe
combined_all_df.sample(5)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION,join_date,Date,Holiday,WeekDay,Month,Day,Year,date,max_temp,min_temp,precipitation,snowfall
16431,481714855,2014-12-24,2908 N DOWNER AVE,PARKED IN EXCESS OF 1 HOUR PROHIBITED,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,2014,2014-12-24,3.2,1.6,1.4,0.0
4917,477892273,2014-06-19,1928 N WATER ST,PARKED WITHIN 4 FEET OF DRIVE OR ALLEY,2014-06-19,2014-06-19,Juneteenth,Thursday,6,19,2014,2014-06-19,16.0,12.8,3.3,0.0
6131,477532215,2014-06-19,789 N JEFFERSON ST,METER PARKING VIOLATION,2014-06-19,2014-06-19,Juneteenth,Thursday,6,19,2014,2014-06-19,16.0,12.8,3.3,0.0
15854,481597163,2014-12-24,1118 N 29TH ST,NIGHT PARKING,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,2014,2014-12-24,3.2,1.6,1.4,0.0
18151,481513410,2014-12-31,3929 N 81ST ST,NIGHT PARKING - WRONG SIDE,2014-12-31,2014-12-31,New Year’s Eve,Wednesday,12,31,2014,2014-12-31,-7.0,-13.8,0.0,0.0


### Feature Engineering

In [37]:
# See what Violation categories can be made out of the data
combined_all_df.value_counts('VIODESCRIPTION')

VIODESCRIPTION
METER PARKING VIOLATION                        4178
NIGHT PARKING - WRONG SIDE                     3969
NIGHT PARKING                                  3776
IMPROPERLY DISPLAYED VEHICLE REGISTRATION      1476
PARKING PROHIBITED BY OFFICIAL SIGN            1062
PARKED IN EXCESS OF 2 HOURS PROHIBITED          975
PARKED LESS THAN 15 FEET FROM CROSSWALK         858
NIGHT PARKING - WINTER RESTRICTED               714
PARKED IN EXCESS OF 1 HOUR PROHIBITED           273
PARKED WITHIN 4 FEET OF DRIVE OR ALLEY          265
PARKED POSTED PRIVATE PROPERTY                  237
TOW-AWAY ZONE (POSTED)                          169
PARKED WITHIN 10 FEET OF FIRE HYDRANT           137
UNREGISTERED/ IMPROPERLY REGISTERED VEHICLE     113
PARKED IN CROSSWALK                              91
PARKED IN EXCESS OF 3 HOURS PROHIBITED           90
RESIDENTIAL PARKING PROGRAM                      89
PARKING IN EXCESS OF 24 HOURS                    84
OBSTRUCTING BUS LOADING ZONE                     

In [51]:
# Feature for  Winter Storm? Inclement Weather?  Threshold can be adjusted 
combined_all_df['is_snow_emergency'] = ((combined_all_df['VIODESCRIPTION'].str.lower().str.contains('snow|emergency')) |      # in-line OR
                                        (combined_all_df['snowfall'] > 3.0)).astype(int)

# Heavy Rain?
combined_all_df['is_heavy_rain'] = (combined_all_df['precipitation'] > 2.0).astype(int)

# Meter Violation?
combined_all_df['is_meter_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('meter').astype(int) # 1 = Meter Violation, 0 = Not a Meter Violation

# Night Violation?
combined_all_df['is_night_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('night').astype(int) # 1 = Night Violation, 0 = Not a Night Violation

# Registration Violation?
combined_all_df['is_registration_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('registration|unregistered').astype(int) # 1 = Registration Violation, 0 = Not a Registration Violation

# Time Violation ?
combined_all_df['is_time_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('excess|hours|time').astype(int) # 1 = Time Violation, 0 = Not a Time Violation

# Weekend Violation?  Could add Friday as well if wanted
combined_all_df['is_weekend_ticket'] = combined_all_df['WeekDay'].str.lower().str.contains('saturday|sunday').astype(int) # 1 = Weekend Violation, 0 = Not a Weekend Violation



combined_all_df.sample(8)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION,join_date,Date,Holiday,WeekDay,Month,Day,...,min_temp,precipitation,snowfall,is_snow_emergency,is_meter_ticket,is_night_ticket,is_registration_ticket,is_time_ticket,is_weekend_ticket,is_heavy_rain
13850,481042833,2014-11-26,2465 N 48TH ST,NIGHT PARKING,2014-11-26,2014-11-26,Thanksgiving Eve,Wednesday,11,26,...,-6.8,0.0,0.00,0,0,1,0,0,0,0
16049,481489433,2014-12-24,2180 S MUSKEGO AVE,NIGHT PARKING - WRONG SIDE,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,...,1.6,1.4,0.00,0,0,1,0,0,0,0
7633,479068715,2014-08-30,1201 S 14TH ST,PARKED LESS THAN 15 FEET FROM CROSSWALK,2014-08-30,2014-08-30,Labor Day Weekend,Saturday,9,30,...,21.3,0.4,0.00,0,0,0,0,0,1,0
1991,475385293,2014-02-14,4267 N 47TH ST,NIGHT PARKING,2014-02-14,2014-02-14,Valentine’s Day,Friday,2,14,...,-8.6,0.0,0.00,0,0,1,0,0,0,0
11298,480814655,2014-11-11,2300 E EUCLID AVE,PARKED LESS THAN 15 FEET FROM CROSSWALK,2014-11-11,2014-11-11,Veterans Day,Tuesday,11,11,...,-0.2,5.5,0.21,0,0,0,0,0,0,1
19183,481716826,2014-12-31,2665 N 95TH ST,NIGHT PARKING - WRONG SIDE,2014-12-31,2014-12-31,New Year’s Eve,Wednesday,12,31,...,-13.8,0.0,0.00,0,0,1,0,0,0,0
16773,481738342,2014-12-24,1030 N MARKET ST,METER PARKING VIOLATION,2014-12-24,2014-12-24,Christmas Eve,Wednesday,12,24,...,1.6,1.4,0.00,0,1,0,0,0,0,0
3650,475331791,2014-02-17,5827 W MEINECKE AVE,NIGHT PARKING - WRONG SIDE,2014-02-17,2014-02-17,Washington's Birthday,Monday,2,17,...,-7.7,8.7,6.30,1,0,1,0,0,0,1
